## Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:

- Role - Identifies the message type(e.g. system, user)
- Content - Representd the actual content of the message (like text, images, audio, documents, etc.)
- Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

### Text Prompts

Text prompts are strings - ideal for straightforward generation tasks where you don't want to retain conversation history.

In [ ]:
model.invoke("What is JavaScript?")

Use text prompts when:
- You have a single, standalone request
- You don't need conversation history
- you want minimal code complexity

### Message Prompt

Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message Types
- System message - Tells the model how to behave and provide context for interactions
- Human message - Represents user input and interactions with the model
- AI message - Responses generated by the model, including text content, tool calls, and metadata
- Tool message - Represents the outputs of tool calls

### System Message

A SystemMessage represent an initial set of instructions that primes the model's behavior. You can use a system message to set the tone, define the model's role, and establish guidelines for responses.

### Human Message

A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content

### AI Message

An AIMessage represents the output of a modal invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can leter access

### Tool Message

For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [ ]:
# Importing All Types of Messages
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a JS expert"), # Here we defining the system
    HumanMessage("Tell me what is array")
]

response = model.invoke(messages)
response.content

In [ ]:
system_msg = SystemMessage("You are a JS expert")

messages = [
    system_msg,
    HumanMessage("Tell me what is array")
]

response = model.invoke(messages)
print(response.content)

In [ ]:
# Detailed Information to the LLM through System Message
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior javascript developer with expertise in JavaScript.
Always give me codes and explanations.
Be concise and precise in your responses.
"""
)

messages = [
    system_msg,
    HumanMessage("Tell me what is array")
]

response = model.invoke(messages)
print(response.content)

In [ ]:
# Message Metadata
human_msg = HumanMessage(
    content="Hello",
    name="Nilay", # Optional: identify the sender
    id="msg_123", # Optional: a unique identifier for the tracing
)

In [ ]:
response = model.invoke([human_msg])
response

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd happy to help you with that question.")

messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("What is JavaScript?"),
    ai_msg, # Insert as if it came from the model
    HumanMessage("What is 2+2?"),
]

response = model.invoke(messages)
print(response.content)

In [ ]:
# To show the MetaData Like Token Details
response.usage_metadata

In [ ]:
from langchain.messages import ToolMessage, AIMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating a message)
ai_message = AIMessage(
    content=[],
    tool_calls=[
        {
            "name": "get_weather",
            "args": {"location": "Kolkata"},
            "id": "nnv6zz3rm",
            "type": "tool_call",
        }
    ],
)

# Execute tool and create result message
weather_result = "Rainy, 28 degrees"
# When tool_message response comes back, we give it to the model
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123",
)

# Continue Conversation
messages = [
    HumanMessage("What is the weather in Kolkata?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]

response = model.invoke(messages)  # Model processes the result

In [ ]:
tool_message
response.content

'<think>\nOkay, the user asked for the weather in Kolkata. I used the get_weather function with the location set to Kolkata. The response came back as "Rainy, 28 degrees". Now I need to present this information clearly.\n\nFirst, I should mention the current weather condition, which is rainy. Then the temperature, which is 28 degrees Celsius. Since it\'s in Kolkata, maybe I should note that it\'s typical for the region, especially during the monsoon season. Also, maybe suggest carrying an umbrella because it\'s raining. Keep it friendly and helpful. Let me check if there\'s any more details needed, but the user only asked for the weather, so sticking to that should be fine.\n</think>\n\nThe current weather in Kolkata is **rainy** with a temperature of **28°C**. It might be a good idea to carry an umbrella! 🌧️'